# GRU Training for PTM Prediction

Trains a Gated Recurrent Unit (GRU) network for protein sequence classification.

## Architecture Overview
- **Input**: One-hot encoded protein sequences (31 amino acids)
- **Embedding**: Optional learned embedding layer
- **GRU Layers**: Bidirectional GRU layers to capture sequential dependencies
- **Attention**: Optional attention mechanism to focus on important positions
- **Dense Layers**: Fully connected layers for classification
- **Output**: 3 binary predictions (one per PTM type)

## Why GRU for Proteins?
- GRUs capture long-range dependencies in sequences
- Bidirectional processing captures context from both directions
- Better than LSTM for shorter sequences (31 residues)
- Attention helps identify critical residues for modification

## Configuration

In [ ]:
# ============================================================================
# CONFIGURATION PARAMETERS - MODIFY THESE AS NEEDED
# ============================================================================

# Set experiment name
EXPERIMENT_NAME = "run01_baseline"

# File paths
TRAIN_FILE = "../output/data_engineered/train_split.csv"
VAL_FILE = "../output/data_engineered/val_split.csv"
OUTPUT_DIR = "../output/gru/"
EXP_DIR = os.path.join(OUTPUT_DIR, EXPERIMENT_NAME)
os.makedirs(EXP_DIR, exist_ok=True)

# Sequence parameters
SEQ_LENGTH = 31  # All sequences are 31 amino acids
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"  # 20 standard amino acids
VOCAB_SIZE = len(AMINO_ACIDS) + 1  # +1 for padding/unknown

# Model architecture
EMBEDDING_DIM = 64  # Learned embedding dimension
GRU_UNITS = [128, 64]  # GRU units per layer
USE_BIDIRECTIONAL = True  # Use bidirectional GRU
USE_ATTENTION = True  # Add attention mechanism
DROPOUT_RATE = 0.3  # Dropout for regularization
RECURRENT_DROPOUT = 0.2  # Dropout in recurrent connections
DENSE_UNITS = [64, 32]  # Dense layer sizes

# Training parameters
BATCH_SIZE = 128
EPOCHS = 50
LEARNING_RATE = 0.001
EARLY_STOPPING_PATIENCE = 10  # Stop if no improvement for N epochs

# Class imbalance handling
USE_CLASS_WEIGHTS = True  # Apply class weights for imbalanced data

# Control CPU usage
N_CORES = 16

print("Configuration loaded:")
print(f"  Sequence length: {SEQ_LENGTH}")
print(f"  Vocab size: {VOCAB_SIZE}")
print(f"  Embedding dim: {EMBEDDING_DIM}")
print(f"  Conv filters: {CONV_FILTERS}")
print(f"  Kernel sizes: {KERNEL_SIZES}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Results will be saved to: {EXP_DIR}")

## Set Usage Variables

In [ ]:
import os

# 1. Set Environment Variables
os.environ["OMP_NUM_THREADS"] = str(N_CORES)
os.environ["TF_NUM_INTRAOP_THREADS"] = str(N_CORES)
os.environ["TF_NUM_INTEROP_THREADS"] = str(N_CORES)

# 2. Import TensorFlow
import tensorflow as tf

# 3. Configure TensorFlow Threads (Must run immediately after import)
tf.config.threading.set_intra_op_parallelism_threads(N_CORES)
tf.config.threading.set_inter_op_parallelism_threads(N_CORES)

print(f"TensorFlow restricted to {N_CORES} threads.")

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import f1_score, roc_auc_score, classification_report
import pickle
import os
import warnings
warnings.filterwarnings("ignore")

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## Load Data

In [ ]:
print("="*80)
print("LOADING DATA")
print("="*80)

train_df = pd.read_csv(TRAIN_FILE)
val_df = pd.read_csv(VAL_FILE)

label_cols = ["S-glutathionylation", "S-nitrosylation", "S-palmitoylation"]

print(f"\nTrain samples: {len(train_df):,}")
print(f"Val samples: {len(val_df):,}")

# Extract sequences and labels
train_sequences = train_df["Sequence"].values
val_sequences = val_df["Sequence"].values

y_train = train_df[label_cols].values.astype(np.float32)
y_val = val_df[label_cols].values.astype(np.float32)

print(f"\nLabel distribution in training:")
for i, label in enumerate(label_cols):
    pos = y_train[:, i].sum()
    pct = pos / len(y_train) * 100
    print(f"  {label}: {int(pos):,} ({pct:.2f}%)")

## Sequence Encoding

In [ ]:
def create_aa_mapping(amino_acids):
    """Create amino acid to integer mapping."""
    aa_to_int = {aa: i+1 for i, aa in enumerate(amino_acids)}  # Start from 1 (0 reserved for padding)
    aa_to_int["X"] = 0  # Unknown/padding
    return aa_to_int

def encode_sequences(sequences, aa_to_int, max_length):
    """Encode sequences as integer arrays."""
    encoded = np.zeros((len(sequences), max_length), dtype=np.int32)
    
    for i, seq in enumerate(sequences):
        for j, aa in enumerate(seq[:max_length]):
            encoded[i, j] = aa_to_int.get(aa, 0)  # 0 for unknown
    
    return encoded

# Create mapping
aa_to_int = create_aa_mapping(AMINO_ACIDS)
print(f"Amino acid mapping created: {len(aa_to_int)} characters")

# Encode sequences
X_train = encode_sequences(train_sequences, aa_to_int, SEQ_LENGTH)
X_val = encode_sequences(val_sequences, aa_to_int, SEQ_LENGTH)

print(f"\nEncoded sequences:")
print(f"  X_train shape: {X_train.shape}")
print(f"  X_val shape: {X_val.shape}")
print(f"  y_train shape: {y_train.shape}")
print(f"  y_val shape: {y_val.shape}")

## Calculate Class Weights

In [ ]:
if USE_CLASS_WEIGHTS:
    class_weights_list = []
    
    for i, label in enumerate(label_cols):
        pos_count = y_train[:, i].sum()
        neg_count = len(y_train) - pos_count
        
        # Weight for class 1 (positive)
        pos_weight = neg_count / pos_count if pos_count > 0 else 1.0
        
        class_weights_list.append({0: 1.0, 1: pos_weight})
        print(f"{label}: pos_weight = {pos_weight:.2f}")
    
    print("\n✓ Class weights calculated")
else:
    class_weights_list = None
    print("Class weights disabled")

## Custom Attention Layer

In [ ]:
class AttentionLayer(layers.Layer):
    """Simple attention mechanism for sequence data."""
    
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)
    
    def build(self, input_shape):
        self.W = self.add_weight(
            name="attention_weight",
            shape=(input_shape[-1], input_shape[-1]),
            initializer="glorot_uniform",
            trainable=True
        )
        self.b = self.add_weight(
            name="attention_bias",
            shape=(input_shape[-1],),
            initializer="zeros",
            trainable=True
        )
        super(AttentionLayer, self).build(input_shape)
    
    def call(self, inputs):
        # inputs shape: (batch, time_steps, features)
        # Compute attention scores
        e = tf.nn.tanh(tf.tensordot(inputs, self.W, axes=1) + self.b)
        e = tf.reduce_sum(e, axis=-1, keepdims=True)  # (batch, time_steps, 1)
        
        # Softmax to get attention weights
        alpha = tf.nn.softmax(e, axis=1)  # (batch, time_steps, 1)
        
        # Weighted sum
        output = tf.reduce_sum(inputs * alpha, axis=1)  # (batch, features)
        
        return output
    
    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

print("✓ Attention layer defined")

## Build GRU Model

In [ ]:
def build_gru_model(
    vocab_size,
    seq_length,
    embedding_dim,
    gru_units,
    use_bidirectional,
    use_attention,
    dense_units,
    dropout_rate,
    recurrent_dropout,
    num_labels
):
    """
    Build a GRU model for protein sequence classification.
    
    Architecture:
    - Embedding layer
    - Stacked (Bidirectional) GRU layers
    - Optional attention mechanism
    - Dense layers
    - Output layer (sigmoid for multi-label)
    """
    inputs = layers.Input(shape=(seq_length,), name="sequence_input")
    
    # Embedding layer
    x = layers.Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        input_length=seq_length,
        name="embedding"
    )(inputs)
    
    # Stacked GRU layers
    for i, units in enumerate(gru_units):
        return_sequences = (i < len(gru_units) - 1) or use_attention
        
        gru = layers.GRU(
            units=units,
            return_sequences=return_sequences,
            dropout=dropout_rate,
            recurrent_dropout=recurrent_dropout,
            name=f"gru_{i+1}"
        )
        
        if use_bidirectional:
            x = layers.Bidirectional(gru, name=f"bi_gru_{i+1}")(x)
        else:
            x = gru(x)
        
        # Batch normalization after each GRU layer
        if return_sequences:
            x = layers.TimeDistributed(
                layers.BatchNormalization(),
                name=f"bn_gru_{i+1}"
            )(x)
        else:
            x = layers.BatchNormalization(name=f"bn_gru_{i+1}")(x)
    
    # Attention mechanism (if enabled)
    if use_attention:
        x = AttentionLayer(name="attention")(x)
    
    # Dropout
    x = layers.Dropout(dropout_rate, name="dropout_1")(x)
    
    # Dense layers
    for i, units in enumerate(dense_units):
        x = layers.Dense(units, activation="relu", name=f"dense_{i+1}")(x)
        x = layers.BatchNormalization(name=f"bn_dense_{i+1}")(x)
        x = layers.Dropout(dropout_rate, name=f"dropout_{i+2}")(x)
    
    # Output layer (sigmoid for multi-label classification)
    outputs = layers.Dense(num_labels, activation="sigmoid", name="output")(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="PTM_GRU")
    
    return model

# Build model
model = build_gru_model(
    vocab_size=VOCAB_SIZE,
    seq_length=SEQ_LENGTH,
    embedding_dim=EMBEDDING_DIM,
    gru_units=GRU_UNITS,
    use_bidirectional=USE_BIDIRECTIONAL,
    use_attention=USE_ATTENTION,
    dense_units=DENSE_UNITS,
    dropout_rate=DROPOUT_RATE,
    recurrent_dropout=RECURRENT_DROPOUT,
    num_labels=len(label_cols)
)

print("\n" + "="*80)
print("MODEL ARCHITECTURE")
print("="*80)
model.summary()

## Compile Model

In [ ]:
# Use binary crossentropy for multi-label classification
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=[
        "binary_accuracy",
        keras.metrics.AUC(name="auc"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall")
    ]
)

print("✓ Model compiled")

## Setup Callbacks

In [ ]:
# Create output directory
os.makedirs(EXP_DIR, exist_ok=True)

# Callbacks
callbacks = [
    # Early stopping
    EarlyStopping(
        monitor="val_loss",
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
        verbose=1
    ),
    
    # Model checkpoint
    ModelCheckpoint(
        filepath=os.path.join(EXP_DIR, "gru_model_best.h5"),
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    
    # Reduce learning rate on plateau
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
]

print("✓ Callbacks configured")

## Train Model

In [ ]:
print("\n" + "="*80)
print("TRAINING GRU MODEL")
print("="*80)

# Calculate sample weights
if USE_CLASS_WEIGHTS:
    sample_weights_train = np.ones(len(y_train))
    for i in range(len(label_cols)):
        pos_mask = y_train[:, i] == 1
        sample_weights_train[pos_mask] = max(
            sample_weights_train[pos_mask],
            class_weights_list[i][1]
        )
else:
    sample_weights_train = None

# Train
history = model.fit(
    X_train,
    y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    sample_weight=sample_weights_train,
    callbacks=callbacks,
    verbose=1
)

print("\n✓ Training complete!")

## Plot Training History

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Loss
axes[0, 0].plot(history.history["loss"], label="Train Loss")
axes[0, 0].plot(history.history["val_loss"], label="Val Loss")
axes[0, 0].set_title("Loss")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].legend()
axes[0, 0].grid(True)

# AUC
axes[0, 1].plot(history.history["auc"], label="Train AUC")
axes[0, 1].plot(history.history["val_auc"], label="Val AUC")
axes[0, 1].set_title("AUC")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("AUC")
axes[0, 1].legend()
axes[0, 1].grid(True)

# Precision
axes[1, 0].plot(history.history["precision"], label="Train Precision")
axes[1, 0].plot(history.history["val_precision"], label="Val Precision")
axes[1, 0].set_title("Precision")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Precision")
axes[1, 0].legend()
axes[1, 0].grid(True)

# Recall
axes[1, 1].plot(history.history["recall"], label="Train Recall")
axes[1, 1].plot(history.history["val_recall"], label="Val Recall")
axes[1, 1].set_title("Recall")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Recall")
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(EXP_DIRR, "gru_training_history.png"), dpi=150)
plt.show()

print("✓ Training history plotted")

## Evaluate Model

In [ ]:
print("\n" + "="*80)
print("EVALUATION ON VALIDATION SET")
print("="*80)

# Get predictions
y_pred_proba = model.predict(X_val, batch_size=BATCH_SIZE)
y_pred = (y_pred_proba > 0.5).astype(int)

print(f"\n{"Label":<30} {"F1":<12} {"AUC":<12}")
print("-" * 54)

results = {}
for i, label in enumerate(label_cols):
    f1 = f1_score(y_val[:, i], y_pred[:, i])
    auc = roc_auc_score(y_val[:, i], y_pred_proba[:, i])
    results[label] = {"f1": f1, "auc": auc}
    print(f"{label:<30} {f1:<12.4f} {auc:<12.4f}")

macro_f1 = np.mean([r["f1"] for r in results.values()])
macro_auc = np.mean([r["auc"] for r in results.values()])

print(f"\n{"Macro Average":<30} {macro_f1:<12.4f} {macro_auc:<12.4f}")

# Detailed classification report
print("\n" + "="*80)
print("DETAILED CLASSIFICATION REPORT")
print("="*80)
for i, label in enumerate(label_cols):
    print(f"\n[{label}]")
    print(classification_report(
        y_val[:, i],
        y_pred[:, i],
        target_names=["Negative", "Positive"],
        digits=4
    ))

## Save Model and Predictions

In [ ]:
print("\n" + "="*80)
print("SAVING MODEL AND PREDICTIONS")
print("="*80)

# Save full model
model.save(os.path.join(EXP_DIR, "gru_model_final.h5"))
print(f"✓ Saved gru_model_final.h5")

# Save validation predictions (for ensemble)
val_pred_df = pd.DataFrame({
    "ID": val_df["ID"],
    "gru_glut_proba": y_pred_proba[:, 0],
    "gru_nitro_proba": y_pred_proba[:, 1],
    "gru_palm_proba": y_pred_proba[:, 2]
})
val_pred_df.to_csv(os.path.join(EXP_DIR, "gru_val_predictions.csv"), index=False)
print(f"✓ Saved gru_val_predictions.csv")

# Save results
results_df = pd.DataFrame(results).T
results_df.to_csv(os.path.join(EXP_DIR, "gru_results.csv"))
print(f"✓ Saved gru_results.csv")

# Save amino acid mapping (needed for test predictions)
with open(os.path.join(EXP_DIR, "aa_to_int.pkl"), "wb") as f:
    pickle.dump(aa_to_int, f)
print(f"✓ Saved aa_to_int.pkl")

print("\n" + "="*80)
print("✓ GRU TRAINING COMPLETE!")
print("="*80)
print(f"\nFinal Performance:")
print(f"  Macro F1: {macro_f1:.4f}")
print(f"  Macro AUC: {macro_auc:.4f}")
print(f"\nNext steps:")
print(f"  1. Share gru_val_predictions.csv with teammates for ensemble")
print(f"  2. Use gru_model_final.h5 for test predictions")
print(f"  3. Compare with CNN - which performs better?")
print(f"  4. Consider hyperparameter tuning if performance is suboptimal")
print("="*80)

## Optional: Predict on Test Data

Uncomment and run this cell when you have test data ready.

In [ ]:
# # Load test data
# test_df = pd.read_csv("../data/test.csv")
# test_sequences = test_df["Sequence"].values
# 
# # Encode test sequences
# X_test = encode_sequences(test_sequences, aa_to_int, SEQ_LENGTH)
# 
# # Predict
# test_pred_proba = model.predict(X_test, batch_size=BATCH_SIZE)
# 
# # Save predictions
# test_pred_df = pd.DataFrame({
#     "ID": test_df["ID"],
#     "gru_glut_proba": test_pred_proba[:, 0],
#     "gru_nitro_proba": test_pred_proba[:, 1],
#     "gru_palm_proba": test_pred_proba[:, 2]
# })
# test_pred_df.to_csv(os.path.join(OUTPUT_DIR, "gru_test_predictions.csv"), index=False)
# print("✓ Saved gru_test_predictions.csv")